In [ ]:

import os
import matplotlib.pyplot as plt
import gc
import torch
import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join("..")))
from pipeline.preprocessing import BackgroundEraser
import glob

from PIL import Image
from torchvision import transforms

import pandas as pd
import re as regex_module
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, fcluster

from ultralytics import YOLO

from pipeline.preprocessing import Repair, Normalizer
from pipeline.feature_learning.vae import VAE, VAEClassifier
from pipeline.clustering import HierarchicalClustering
from pipeline.evaluation import EvaluationWithGroundTruth

# PIPELINE

In [ ]:
run_id = "run_2026-09-04_class" 

## Preprocessing

### Shard selection and  background cleaning

In [ ]:

#image_paths = glob.glob("/caa/Homes01/alaczko/Diploma/Diploma/data/raw/00_Selection ML/2012_Survey Gladiatorenschule/**/*.jpg", recursive=True)
image_path= "/caa/Homes01/alaczko/Diploma/Diploma/data/raw"
 
br=BackgroundEraser(input_path=image_path, batch_size=32, num_workers=16, run_id=run_id)
#todo "detected 6171 shards, but erased len is 2800"   
erased=br.run()

### Resize & Crop

In [ ]:
crop_step = Normalizer(input_path=erased, background_color=(255, 255, 255), run_id=run_id)

cropped= crop_step.run()

### Enhancing Picture Quality

In [ ]:
repair_step = Repair(
    input_path=cropped,
    max_distance=20,
    output_thickness=3,   # ← ez az új paraméter
    run_id=run_id
)

result_path = repair_step.run()

### Feature Learning

In [ ]:
result_path="/caa/Homes01/alaczko/Diploma/Diploma/data/pipeline/run_2026-09-01/PreprocessingStep/Repair"

#result_path = "/caa/Homes01/alaczko/Diploma/Diploma/data/rocopot_rim_256"

In [ ]:
vae = VAE(input_path=result_path, f=4, n=3, s=1, k=128, run_id=run_id)  # k=128 latent dimension
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vae.to(device)

results = vae.run('train_model', epochs=1000, lr=1e-6, batch_size=128)

torch.save(vae.state_dict(), "../models/feature_learning/legion.pt")

In [ ]:
results = vae.extract_features( batch_size=128)

In [ ]:
results

In [ ]:
results.to_csv("../data/pipeline/default_run/features.csv", index=False)

#### Testing Feature Learning

In [ ]:
vae= VAE(input_path="/caa/Homes01/alaczko/Diploma/Diploma/data/rocopot_rim_256", f=4, n=3, s=1, k=128)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vae.to(device)

vae.load_state_dict(torch.load("../models/feature_learning/roco_all_og.pt", map_location=device))

#making sure the formula is correct
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])

img_path = "/caa/Homes01/alaczko/Diploma/Diploma/data/test-roco.png"
original_img = Image.open(img_path)
input_tensor = transform(original_img).unsqueeze(0).to(device) #adding batch dim

with torch.no_grad():
    reconstructed, mu, sigma = vae(input_tensor)

orig_np = input_tensor.squeeze().cpu().numpy()
recon_np = reconstructed.squeeze().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(orig_np, cmap='gray', vmin=0, vmax=1)
axes[0].set_title("Original")
axes[0].axis('off')

axes[1].imshow(recon_np, cmap='gray', vmin=0, vmax=1)
axes[1].set_title("Reconstructed")
axes[1].axis('off')

plt.tight_layout()
plt.show()

### Classifier

In [ ]:
result_path="/caa/Homes01/alaczko/Diploma/Diploma/data/rocopot_rim_256"


In [ ]:
vae= VAE(input_path="/caa/Homes01/alaczko/Diploma/Diploma/data/rocopot_rim_256", f=4, n=3, s=1, k=128)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vae.to(device)

vae.load_state_dict(torch.load("../models/feature_learning/roco_all_og.pt", map_location=device))


In [ ]:
from pipeline.feature_learning import VAEClassifier

labels_path = "/caa/Homes01/alaczko/Diploma/Diploma/data/rocopot_results.csv"

classifier = VAEClassifier(
    input_path=result_path,          # a VAE bemeneti képeinek elérési útja
    vae=vae,                         # a fentebb betanított VAE
    num_classes=28,
    labels_path=labels_path,
    freeze_encoder=True,             # először csak a classifier tanul
    output_path="../models/feature_learning/vae_classifier",
    run_id=run_id,
).to(device)

history = classifier.run(
    epochs=200,
    lr=1e-4,
    batch_size=128,
    validation_size=0.2,
    num_workers=16,
)

history

### Clustering

In [ ]:
df = pd.read_csv("../data/pipeline/roco_run/features.csv")

In [ ]:
import numpy as np
import ast

def parse_feature_string(s):
    if isinstance(s, str):
        cleaned = s.strip('[]')
        # tetszőleges mennyiségű whitespace -> egyetlen szóköz
        cleaned = regex_module.sub(r'\s+', ' ', cleaned).strip()
        return np.fromstring(cleaned, sep=' ')
    return np.asarray(s)

In [ ]:
df['features'] = df['features'].apply(parse_feature_string)

In [ ]:
X = np.vstack(df['features'].values)

# 2. inaktív / szinte nulla varianciájú dimenziók kiszűrése (opcionális, de ajánlott)
variances = X.var(axis=0)
active_dims = variances > 1e-3   # threshold-öt finomítsd a saját adatodra
X_active = X[:, active_dims]

# 3. standardizálás (dimenziónkénti z-score)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_active)

df["features_non_scaled"] = df['features'] 

# 4. opcionálisan visszaírhatod a df-be, hogy egyben tudd kezelni
df['features'] = list(X_scaled)

df.to_csv("../data/pipeline/roco_run/features_scaled.csv", index=False)

In [ ]:
df.head()

In [ ]:
clustering_step = HierarchicalClustering(
    input_path="../data/pipeline/roco_run/features_scaled.csv",
    cluster_num=55
)
# result= clustering_step.run()


#result = clustering_step.compare_all()

In [ ]:
result= clustering_step.run()

In [ ]:
np.unique(result, return_counts=True)

In [ ]:
df["pred"] = list(result)

In [ ]:
df["filename"] = df["filename"].str.rsplit(".", n=1).str[0]

In [ ]:
roco_labels = pd.read_csv("/caa/Homes01/alaczko/Diploma/Diploma/data/rocopot_results.csv")

In [ ]:
new_df = pd.merge(
    df,
    roco_labels,
    left_on="filename",
    right_on="ID",
    how="inner"
)

In [ ]:
new_df = new_df.dropna(subset=["pred", "TypeRev"]).reset_index(drop=True)

In [ ]:
new_df

In [ ]:
eval_step = EvaluationWithGroundTruth(y_true=new_df["TypeRev"].values, y_pred=new_df["pred"].values)

In [ ]:
eval_step.run()

In [ ]:
features_path = "../data/pipeline/roco_run/features_scaled.csv"

cluster_numbers = range(10, 100)
metric_results = []

# A features.csv sorrendjében találhatók a klaszterezés bemenetei
features_df = pd.read_csv(features_path)

features_df["filename"] = (
    features_df["filename"]
    .astype(str)
    .str.strip()
    .str.rsplit(".", n=1)
    .str[0]
)


In [ ]:
for cluster_num in cluster_numbers:
    clustering_step = HierarchicalClustering(
        input_path=features_path,
        cluster_num=cluster_num
    )

    predicted_labels = clustering_step.run()

    predictions_df = features_df[["filename"]].copy()
    predictions_df["pred"] = predicted_labels

    # Az új klasztercímkék hozzárendelése a meglévő dataframe-hez
    current_df = pd.merge(
        predictions_df,
        roco_labels,
        left_on="filename",
        right_on="ID",
        how="inner"
    )

    current_df = current_df.dropna(subset=["pred", "TypeRev"]).reset_index(drop=True)

    y_true = current_df["TypeRev"].to_numpy()
    y_pred = current_df["pred"].to_numpy()


    eval_step = EvaluationWithGroundTruth(y_true=y_true, y_pred=y_pred)

    eval_step.run()

    metric_results.append({
        "cluster_num": cluster_num,
        "ACC": eval_step.acc,
        "ARI": eval_step.ari,
        "NMI": eval_step.nmi,
        "sample_count": len(current_df),
        "predicted_cluster_count": current_df["pred"].nunique(),
    })

metrics_df = pd.DataFrame(metric_results)
metrics_df

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    metrics_df["cluster_num"],
    metrics_df["ACC"],
    marker="o",
    label="ACC"
)

plt.plot(
    metrics_df["cluster_num"],
    metrics_df["ARI"],
    marker="o",
    label="ARI"
)

plt.plot(
    metrics_df["cluster_num"],
    metrics_df["NMI"],
    marker="o",
    label="NMI"
)

plt.axvline(
    28,
    color="red",
    linestyle="--",
    label="28 archaeological labels"
)

plt.xlabel("Number of predicted clusters")
plt.ylabel("Metric value")
plt.title("ARI and NMI by number of clusters")
plt.xticks(range(10, 61, 5))
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
new_df

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# df-ben legyen egy 'pred' (predikált cluster) és egy 'true_label' (ground truth) oszlop
contingency = pd.crosstab(new_df['TypeRev'], new_df['pred'])

plt.figure(figsize=(14, 10))
sns.heatmap(contingency, annot=False, cmap='viridis', cbar_kws={'label': 'count'})
plt.xlabel('Predicted cluster')
plt.ylabel('Ground truth type')
plt.title('Cluster-Ground Truth contingency matrix')
plt.tight_layout()
plt.show()